# Fetch the atlas data set

This notebook gets every PPMI file that the atlas's worked example uses (`atlas/index.html`, sections 5 and 7) from the LONI Image and Data Archive (IDA), with your own IDA account, into your data folder: `~/ppmi`, or the folder named by `PPMI_DATA`. Run it top to bottom on your own machine. Each step lists what to get, opens the IDA page, waits while you download, then files what arrived and checks it. Run it again at any time: it asks only for what is still missing.

The data set is the three sets marked in the `project` column of `code/catalog.csv`:

- **core**: the twelve tables that `audit.ipynb` needs.
- **pipeline**: three tables that the clean layer also reads: MoCA, the archived Invicro DaTscan ratios and the Xing MRI acquisition metadata.
- **recommended**: what section 7 of the atlas rates *must* or *should*: the medication log, the rest of the MDS-UPDRS, diagnosis history, genetic status, biospecimen results including seed amplification, the Curated Data Cut, FreeSurfer and MRI quality tables, image searches, and the documents that explain them.

IDA has no programming interface for study data, so you tick files and save searches in the browser, and the code does the rest. The data use agreement forbids sharing participant-level data and applying AI tools to it, so this notebook prints file names and counts only. Keep the data folder outside the repository.

In [ ]:
import sys
sys.path.append("../code")

import shutil
import warnings
from collections import Counter
from pathlib import Path

import pandas as pd

import data
import ida

pd.set_option("display.max_rows", 200, "display.max_colwidth", 120)

OPEN_IDA = True   # open each IDA page and wait while you download; False only files and checks what is already there
EXTRA = []        # more datasets, by their name in code/catalog.csv, such as "SCOPA-AUT"
URL_LIST = None   # step 4: the URL list from IDA's Advanced Download, such as "~/Downloads/urls.txt"

folder = ida.root()
catalog = data.catalog()
unknown = sorted(set(EXTRA) - set(catalog.dataset))
if unknown:
    print("Not in code/catalog.csv, so not fetched:", ", ".join(unknown))
print("Data folder:", folder.as_posix().replace(Path.home().as_posix(), "~", 1))


def check(notes=False, searches=False):
    """Load the data folder without the clean layer and line it up with the data set, by file name."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        p = data.PPMI(clean=False)
    for note in p.notes if notes else []:
        print(note)
    for warning in caught if searches else []:
        if str(warning.message).startswith("Skipped"):
            print(warning.message)
    inventory = p.inventory()
    return p, inventory[((inventory.project != "") | inventory.dataset.isin(EXTRA)) & (inventory.format != "pdf")]


def tick(rows):
    """List datasets the way IDA's Study Data page groups them."""
    for (group, subgroup), items in rows.groupby(["group", "subgroup"], sort=False):
        print(f"{group} > {subgroup}")
        for dataset in items.dataset:
            print(f"    [ ] {dataset}")


def wait(page, message):
    if OPEN_IDA:
        ida.browse(page)
        input(f"{message} Then press Enter here.")

## 1. The data set

One row per dataset, with where it sits in IDA. To add the *could* items of section 7, put their names in `EXTRA`, for example `"SCOPA-AUT"` or `"University of Pennsylvania Smell Identification Test (UPSIT)"`, and run the notebook again.

In [ ]:
data_set = catalog[(catalog.project != "") | catalog.dataset.isin(EXTRA)]
data_set.sort_values("project", kind="stable")[["project", "group", "subgroup", "dataset", "format", "notes"]]

## 2. Study data

The list below is what is still missing, grouped as IDA's *Download Study Data* page groups it in its left-hand panel. The next cell opens that page in your browser. Sign in, open each group or type a dataset's name into the search box, tick the datasets and click *Download*. Several ticked files arrive as one zip. Tick the PDF *Method for calculating Levodopa Equivalent Dose* under *Study Docs* too: it explains the doses in the medication log. The [PPMI Data User Guide](https://www.ppmi-info.org/sites/default/files/docs/PPMI%20Data%20User%20Guide.pdf) is public.

When the download has finished, press Enter in the box that appears. `ida.collect()` then files what arrived in your Downloads folder:

- study tables and the Curated Data Cut workbook go to `study/`, keeping IDA's names;
- PDFs that came in the same zip go to `docs/`;
- each zip is kept in `downloads/`;
- files that did not come from IDA stay where they are.

If your browser unzips downloads, as Safari does by default, run `ida.collect("~/Downloads/<the unzipped folder>")`. If the notebook cannot read your Downloads folder, set `IDA_DOWNLOADS` to the folder you download into.

In [ ]:
p, wanted = check()
missing = wanted.query("downloaded == '' and group != 'Image Collections'")
if len(missing):
    tick(missing)
else:
    print("Every study table is here.")
print("\nIn docs/:", ", ".join(sorted(path.name for path in (folder / "docs").glob("*.pdf"))) or "nothing yet")

In [ ]:
if len(missing):
    wait("study", "Sign in, tick the datasets listed above and click Download. Wait for the download to finish.")
ida.collect()

Still missing after this round, with notes on any file that could not be read. Run the two cells above again for anything left. If a table you downloaded is still listed, IDA named its file differently: step 5 lists it under *not in the catalogue*.

In [ ]:
p, wanted = check(notes=True)
missing = wanted.query("downloaded == '' and group != 'Image Collections'")
print(f"{len(missing)} study tables missing")
tick(missing)

## 3. Image searches

An image search lists one scan per row with its participant, visit, modality and image ID. It ties scans to visits, and `audit.ipynb` counts scans from it. In IDA's *Search*, choose *Advanced Search* and run three searches, each with Research Group *PD* and the visits *Screening*, *Baseline*, *Month 12*, *Month 24*, *Month 36*, *Month 48* and *Month 60*:

| Search | Criteria |
|---|---|
| DaTscan | Modality *SPECT* |
| T1-weighted MRI | Modality *MRI*, Weighting *T1*, Acquisition Type *3D* |
| Diffusion MRI | Modality *DTI* |

Show at least Subject ID, Visit, Modality, Image ID, Description, Study Date and Research Group in the results, because the CSV keeps only the columns shown. Save each search with *CSV Download*. The next cell opens the search page if a modality is missing, waits, and files the CSVs into `imaging/`, where they become the `images` table.

In [ ]:
p, wanted = check()
modalities = set(p.sql("SELECT DISTINCT modality FROM images").modality)
if not {"SPECT", "MRI", "DTI"} <= modalities:
    wait("search", "Run the three searches and save each with CSV Download.")
    display(ida.collect())
p, wanted = check(searches=True)
p.sql("SELECT modality, count(DISTINCT PATNO) AS participants, count(*) AS images FROM images GROUP BY ALL ORDER BY 1")

## 4. Raw images (optional)

Steps 2 and 3 give the derived measures: DaTscan binding ratios, grey-matter volume, FreeSurfer volumes and cortical thickness. They are enough for the worked example's main analysis. Raw scans are for a regional analysis of your own, which section 7 rates *should*. They are large: one T1 series is tens of megabytes and a DTI series more, so a cohort runs to tens of gigabytes. Compare the collection's size in IDA with the free space printed below.

To get them, open the T1 or DTI search results in IDA, select the images and choose *Add to Collection*. Under *Data Collections*, choose *Advanced Download* and save the list of URLs it offers. Put that file's path in `URL_LIST` in the setup cell and run the cell below. `ida.fetch` downloads every part, resumes a broken transfer, skips parts it already has and unpacks the images into `images/`, where they become the `image_files` table. A small collection can come as one zip from *1-Click Download*; `ida.collect()` files that.

In [ ]:
print(f"Free space on the data folder's disk: {shutil.disk_usage(folder if folder.exists() else Path.home()).free / 1e9:,.0f} GB")
if URL_LIST:
    display(ida.fetch(URL_LIST))
    p, wanted = check()
p.sql("SELECT format, count(*) AS series, count(DISTINCT PATNO) AS participants, round(sum(bytes) / 1e9, 2) AS gigabytes FROM image_files GROUP BY ALL")

## 5. Check everything

This loads the data folder the way the other notebooks do, clean layer included, and reports what is here, what is missing and what matches nothing in the catalogue. If a dataset you downloaded shows as missing, IDA named its file differently. Find it under *not in the catalogue* and write its name without the date, such as `Participant_Status`, in the dataset's `file` column in `code/catalog.csv`.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    p = data.PPMI()
inventory = p.inventory()
wanted = inventory[((inventory.project != "") | inventory.dataset.isin(EXTRA)) & (inventory.format != "pdf")]
here = wanted.query("downloaded != ''")
print(f"{len(here)} of {len(wanted)} datasets are here.")
for note in p.notes + [str(warning.message) for warning in caught if str(warning.message).startswith("Skipped")]:
    print(note)
here.sort_values(["project", "group", "dataset"])[["project", "group", "dataset", "downloaded", "rows", "participants"]]

In [ ]:
missing = wanted.query("downloaded == ''")
print(f"Missing: {len(missing)}")
tick(missing)
unmatched = inventory.query("group == 'Other downloads'")
print("\nNot in the catalogue:", ", ".join(unmatched.downloaded) or "none")
print("In docs/:", ", ".join(sorted(path.name for path in (folder / "docs").glob("*.pdf"))) or "nothing yet")

IDA updates tables between downloads, so tables downloaded on different days may disagree. For a final analysis, download every table on the same day.

In [ ]:
dates = Counter(date.date() for date in p.releases.values() if date.year > 1)
print("Download dates:", ", ".join(f"{day:%d %b %Y} ({count} tables)" for day, count in sorted(dates.items())) or "none")

The clean layer reads its tables by name. `status` is `loaded` for each table it found and `missing` for the rest.

In [ ]:
if p.clean_error is not None:
    print(f"The clean layer was not built ({type(p.clean_error).__name__}). p.clean_error has the cause.")
if "clean.source_manifest" in p.tables():
    display(p.sql("SELECT source_key AS source, filename AS file, status, input_rows AS rows FROM clean.source_manifest ORDER BY status, source"))

## Next

- `dataset.ipynb` shows everything IDA offers next to what you now have, and the SQL interface.
- `schema.ipynb` maps how the tables join.
- `audit.ipynb` reruns the preliminary audit on this download.
- `python code/data.py --export ~/ppmi/ppmi-2026-11-18.duckdb`, with the date of your download, saves both layers to one DuckDB file kept beside the data.

Genetic status covers pathogenic variants such as LRRK2, GBA and SNCA, and APOE has its own table. MAPT H1 and COMT, which section 7 also names, need genotypes from IDA's *Genetic Data* page; those files are large, so check their sizes there first.